# SQuAD v1.1 — QA trích xuất span — hardcore

Model scratch **chưa train**; dự đoán chỉ minh họa inference, không phải kết quả chất lượng. Bạn tự thiết kế tiền xử lý, thay model và viết training sau khi đọc luồng vào/ra này.

## 1. Load dữ liệu cơ bản

Tập inference: **official validation SQuAD v1.1 có nhãn, dùng làm held-out như hai bản đầy đủ; không phải hidden test**. Nguồn/revision trong SOURCE/DATASET_ID. Dữ liệu thô được giữ nguyên; thao tác nhóm/sắp ID chỉ để giữ giao thức mẫu của hai bản đầy đủ, không fit đặc trưng.

Chạy từ thư mục dự án, sửa DATA_DIR trước Run All. DATA_MODE="native" đọc nguồn đã khai báo; DATA_MODE="local_jsonl" đọc bản xuất **đúng nguồn**, schema: validation.jsonl: id:str, context:str, question:str; không thay đổi context hoặc dùng answers trong forward. Đường dẫn ảnh trong JSONL phải hợp lệ từ thư mục chạy. Nếu dùng nguồn HF cần cài datasets; COCO cần torchvision/pycocotools; TFDS cần kernel TensorFlow tương thích. Không tự thay dataset khi thiếu dữ liệu.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import random
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from PIL import Image

SEED=42
BATCH_SIZE=16
MAX_INPUT_BYTES=4096  # Giới hạn đầu vào cố định của model; không đổi số mẫu inference.
MAX_OUTPUT_TOKENS=32
IMAGE_SIZE=64
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_MODE="native"  # Chọn local_jsonl rõ ràng để dùng bản xuất đúng dataset/schema bên dưới.

DATA_DIR=Path('data/qa_squad_v11')
CACHE_DIR=Path("artifacts/qa_squad_v11/hardcore/data_cache")
INFERENCE_SPLIT='held_out'
SOURCE={'row': 3, 'task': 'Extractive QA', 'dataset': 'SQuAD v1.1', 'tfds_builder': 'squad/v1.1:3.0.0', 'original_hint': 'squad/v1.1.0 (đã sửa config thành v1.1, builder version3.0.0)', 'url': 'https://www.tensorflow.org/datasets/catalog/squad'}
DATASET_ID='SQuAD v1.1'
DATASET_REVISION=None


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Cung cấp JSONL từ đúng dataset/revision/schema đã ghi; không dùng dữ liệu giả thay thế.")
    with path.open(encoding="utf-8") as handle:
        rows=[json.loads(line) for line in handle if line.strip()]
    if not rows:raise ValueError(f"Tập dữ liệu rỗng: {path}")
    return rows

In [ ]:
if DATA_MODE=="local_jsonl":rows=read_jsonl(DATA_DIR/"validation.jsonl")
elif DATA_MODE=="native":
    path=DATA_DIR/"dev-v1.1.json"
    if not path.is_file():raise FileNotFoundError(f"Cần {path} từ https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json hoặc local_jsonl đúng nguồn.")
    document=json.loads(path.read_text(encoding="utf-8"))
    if document.get("version")!="1.1":raise ValueError("Cần SQuAD1.1, không dùng v2.")
    rows=[{**qa,"context":paragraph["context"]} for article in document["data"] for paragraph in article["paragraphs"] for qa in paragraph["qas"]]
else:raise ValueError("DATA_MODE phải là native hoặc local_jsonl.")
records=sorted([{"id":row["id"],"context":row["context"],"question":row["question"]} for row in rows],key=lambda row:row["id"])

assert records, "Không có mẫu test/validation để inference."
assert len({row["id"] for row in records})==len(records), "ID đầu vào bị trùng."
print("Tập inference:",INFERENCE_SPLIT,"; số mẫu:",len(records))

## 2. Định nghĩa một model và inference trực tiếp

Các lớp học đều khởi tạo ngẫu nhiên. Chỉ chuyển đầu vào tối thiểu bên trong model; không có tokenizer/vocabulary fit, hàm tiền xử lý riêng, loss, optimizer, training hay checkpoint. Giới hạn chiều đầu vào/đầu ra ở phần cấu hình là cố định và chỉ phục vụ model minh họa; toàn bộ mẫu của split vẫn được inference. Không truyền gold target vào forward.

In [ ]:
class RawSpanModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(257,16)
        self.head=nn.Linear(32,2)
    def forward(self,rows):
        results=[]
        for row in rows:
            context=row["context"]
            if not context:raise ValueError("Context QA rỗng.")
            # Giới hạn 4096 ký tự context được công khai; output là substring của chuỗi gốc.
            ids=[ord(ch)%256 for ch in context[:MAX_INPUT_BYTES]]
            q=list(row["question"].encode("utf-8")[:MAX_INPUT_BYTES]) or [256]
            device=self.embedding.weight.device
            hidden=self.embedding(torch.tensor(ids,device=device))
            question=self.embedding(torch.tensor(q,device=device)).mean(0).expand(len(ids),-1)
            logits=self.head(torch.cat([hidden,question],-1))
            start=int(logits[:,0].argmax().item())
            stop=min(len(ids),start+MAX_OUTPUT_TOKENS)
            end=start+int(logits[start:stop,1].argmax().item())+1
            results.append(context[start:end])
        return results

In [ ]:
model=RawSpanModel().to(DEVICE)
model.eval()
predictions=[]
with torch.inference_mode():
    for start in range(0,len(records),BATCH_SIZE):
        batch=records[start:start+BATCH_SIZE]
        inputs=[{"context":row["context"],"question":row["question"]} for row in batch]
        outputs=model(inputs)
        assert len(outputs)==len(batch), "Số dự đoán trong batch không khớp."
        predictions.extend([{'id':row["id"],'prediction':output} for row,output in zip(batch,outputs)])

assert len(predictions)==len(records)>0
assert [row['id'] for row in predictions]==[row["id"] for row in records]
assert len({row['id'] for row in predictions})==len(records)
assert all(isinstance(row['prediction'],str) for row in predictions)

print(f"Đã inference {len(predictions)} mẫu trên {INFERENCE_SPLIT} bằng model scratch CHƯA TRAIN.")
print("Đầu ra minh họa thực thi, chưa đo chất lượng; cần tự xây dựng tiền xử lý, model và training.")
print(predictions[:2])
# File dự đoán chỉ được tạo khi người dùng chạy notebook; không cần checkpoint/helper.
PREDICTION_PATH=Path("artifacts/qa_squad_v11/hardcore/predictions.json")
PREDICTION_PATH.parent.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")